# [9665] Content-based Recommender
Data file:
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Seattle_hotels.csv

In [ ]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 02/09/25 13:32:54


### Import libraries

In [ ]:
import pandas as pd
import re
import string
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Load data

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Seattle_hotels.csv')
df.shape

(152, 3)

### Examine data

In [ ]:
pd.set_option('max_colwidth', None)

In [ ]:
df.head()

,name,address,desc
0,Hilton Garden Seattle Downtown,"1821 Boren Avenue, Seattle Washington 98101 USA","Located on the southern tip of Lake Union, the Hilton Garden Inn Seattle Downtown hotel is perfectly located for business and leisure. \nThe neighborhood is home to numerous major international companies including Amazon, Google and the Bill & Melinda Gates Foundation. A wealth of eclectic restaurants and bars make this area of Seattle one of the most sought out by locals and visitors. Our proximity to Lake Union allows visitors to take in some of the Pacific Northwest's majestic scenery and enjoy outdoor activities like kayaking and sailing. over 2,000 sq. ft. of versatile space and a complimentary business center. State-of-the-art A/V technology and our helpful staff will guarantee your conference, cocktail reception or wedding is a success. Refresh in the sparkling saltwater pool, or energize with the latest equipment in the 24-hour fitness center. Tastefully decorated and flooded with natural light, our guest rooms and suites offer everything you need to relax and stay productive. Unwind in the bar, and enjoy American cuisine for breakfast, lunch and dinner in our restaurant. The 24-hour Pavilion Pantry? stocks a variety of snacks, drinks and sundries."
1,Sheraton Grand Seattle,"1400 6th Avenue, Seattle, Washington 98101 USA","Located in the city's vibrant core, the Sheraton Grand Seattle provides a gateway to the diverse sights and sounds of the Pacific Northwest. Step out of our front doors to find gourmet dining and bars, world-class shopping, exciting entertainment, and iconic local attractions including the Pike Place Market, Space Needle and Chihuly Garden & Glass Museum. As one of only seven Sheraton hotels in North America to earn the esteemed Grand designation, guests can book confidently knowing they?re receiving the highest benchmark on product and service offerings available. Experience our recently completed multimillion-dollar transformation featuring all new guest rooms, an expanded Sheraton Club Lounge, and modern meeting & event spaces. Gather in our stylish new lobby and enjoy our private art collection featuring local artists while enjoying your favorite beverage from Starbucks. The Sheraton Grand features several dining options including Loulay Kitchen & Bar by James Beard award winning chef Thierry Rautureau."
2,Crowne Plaza Seattle Downtown,"1113 6th Ave, Seattle, WA 98101","Located in the heart of downtown Seattle, the award-winning \nCrowne Plaza Hotel Seattle ? Downtown offers an exceptional blend of service, style and comfort. You?ll notice Cool, Comfortable and Unconventional touches that set us apart as soon as you step inside. Marvel at stunning views of the city lights while relaxing in our new Sleep Advantage? Beds. Enjoy complimentary wireless Internet throughout the hotel and amenities to help you relax like our Temple Spa? Sleep Tight Amenity kits featuring lavender spray and lotions to help you rejuvenate and unwind. Enjoy an invigorating workout at our 24-hour fitness center, get dining suggestions from our expert concierge or savor sumptuous cuisine at our Regatta Bar & Grille restaurant where you can enjoy Happy Hour in our lounge daily from 4pm - 7pm and monthly drink specials. Come and experience all that The Emerald City has to offer with us!"
3,Kimpton Hotel Monaco Seattle,"1101 4th Ave, Seattle, WA98101","What?s near our hotel downtown Seattle location? The better \nquestion might be what?s not nearby. In addition to being one of the hotels near Pike Place Market, here?s just a small sampling of the rest. Columbia Center, whose Sky View Observatory on the 73rd floor is the tallest public viewing area west of the Mississippi Historic 5th Avenue Theatre, home to musical productions Seattle Central Library, an architectural marvel. Within half a mile: The must-see Pike Place Market, which houses the original Starbucks Pioneer Square, Seattle?s original downtown. Seattle Art Museum. 

### Prepare data

In [ ]:
# Drop column address
df.drop(['address'], axis=1, inplace=True)
df.head()

,name,desc
0,Hilton Garden Seattle Downtown,"Located on the southern tip of Lake Union, the Hilton Garden Inn Seattle Downtown hotel is perfectly located for business and leisure. \nThe neighborhood is home to numerous major international companies including Amazon, Google and the Bill & Melinda Gates Foundation. A wealth of eclectic restaurants and bars make this area of Seattle one of the most sought out by locals and visitors. Our proximity to Lake Union allows visitors to take in some of the Pacific Northwest's majestic scenery and enjoy outdoor activities like kayaking and sailing. over 2,000 sq. ft. of versatile space and a complimentary business center. State-of-the-art A/V technology and our helpful staff will guarantee your conference, cocktail reception or wedding is a success. Refresh in the sparkling saltwater pool, or energize with the latest equipment in the 24-hour fitness center. Tastefully decorated and flooded with natural light, our guest rooms and suites offer everything you need to relax and stay productive. Unwind in the bar, and enjoy American cuisine for breakfast, lunch and dinner in our restaurant. The 24-hour Pavilion Pantry? stocks a variety of snacks, drinks and sundries."
1,Sheraton Grand Seattle,"Located in the city's vibrant core, the Sheraton Grand Seattle provides a gateway to the diverse sights and sounds of the Pacific Northwest. Step out of our front doors to find gourmet dining and bars, world-class shopping, exciting entertainment, and iconic local attractions including the Pike Place Market, Space Needle and Chihuly Garden & Glass Museum. As one of only seven Sheraton hotels in North America to earn the esteemed Grand designation, guests can book confidently knowing they?re receiving the highest benchmark on product and service offerings available. Experience our recently completed multimillion-dollar transformation featuring all new guest rooms, an expanded Sheraton Club Lounge, and modern meeting & event spaces. Gather in our stylish new lobby and enjoy our private art collection featuring local artists while enjoying your favorite beverage from Starbucks. The Sheraton Grand features several dining options including Loulay Kitchen & Bar by James Beard award winning chef Thierry Rautureau."
2,Crowne Plaza Seattle Downtown,"Located in the heart of downtown Seattle, the award-winning \nCrowne Plaza Hotel Seattle ? Downtown offers an exceptional blend of service, style and comfort. You?ll notice Cool, Comfortable and Unconventional touches that set us apart as soon as you step inside. Marvel at stunning views of the city lights while relaxing in our new Sleep Advantage? Beds. Enjoy complimentary wireless Internet throughout the hotel and amenities to help you relax like our Temple Spa? Sleep Tight Amenity kits featuring lavender spray and lotions to help you rejuvenate and unwind. Enjoy an invigorating workout at our 24-hour fitness center, get dining suggestions from our expert concierge or savor sumptuous cuisine at our Regatta Bar & Grille restaurant where you can enjoy Happy Hour in our lounge daily from 4pm - 7pm and monthly drink specials. Come and experience all that The Emerald City has to offer with us!"
3,Kimpton Hotel Monaco Seattle,"What?s near our hotel downtown Seattle location? The better \nquestion might be what?s not nearby. In addition to being one of the hotels near Pike Place Market, here?s just a small sampling of the rest. Columbia Center, whose Sky View Observatory on the 73rd floor is the tallest public viewing area west of the Mississippi Historic 5th Avenue Theatre, home to musical productions Seattle Central Library, an architectural marvel. Within half a mile: The must-see Pike Place Market, which houses the original Starbucks Pioneer Square, Seattle?s original downtown. Seattle Art Museum. Fantastic shopping, including the flagship Nordstrom, Nordstrom Rack, Macy?s, Columbia Sportswear, Louis Vuitton, Arcteryx, and oodles of independent boutiques. The Great Whe

#### Clean column hotel descriptions
1) remove punctuation  
2) lowercase text  
3) either stem or lemmatize text

In [ ]:
# Instantiate Porter stemmer
ps = nltk.PorterStemmer()

In [ ]:
# Create function to clean_text
def clean_text(text):
    text = "".join([word.lower() for word in text if word not in string.punctuation])
    tokens = re.split('\W+', text)
    text = [ps.stem(word) for word in tokens]
    text_2 = ' '.join(word for word in text)
    return text_2

In [ ]:
# Apply clean_text function to clean hotel description field
df['desc_clean'] = df['desc'].apply(clean_text)
df.head()

,name,desc,desc_clean
0,Hilton Garden Seattle Downtown,"Located on the southern tip of Lake Union, the Hilton Garden Inn Seattle Downtown hotel is perfectly located for business and leisure. \nThe neighborhood is home to numerous major international companies including Amazon, Google and the Bill & Melinda Gates Foundation. A wealth of eclectic restaurants and bars make this area of Seattle one of the most sought out by locals and visitors. Our proximity to Lake Union allows visitors to take in some of the Pacific Northwest's majestic scenery and enjoy outdoor activities like kayaking and sailing. over 2,000 sq. ft. of versatile space and a complimentary business center. State-of-the-art A/V technology and our helpful staff will guarantee your conference, cocktail reception or wedding is a success. Refresh in the sparkling saltwater pool, or energize with the latest equipment in the 24-hour fitness center. Tastefully decorated and flooded with natural light, our guest rooms and suites offer everything you need to relax and stay productive. Unwind in the bar, and enjoy American cuisine for breakfast, lunch and dinner in our restaurant. The 24-hour Pavilion Pantry? stocks a variety of snacks, drinks and sundries.",locat on the southern tip of lake union the hilton garden inn seattl downtown hotel is perfectli locat for busi and leisur the neighborhood is home to numer major intern compani includ amazon googl and the bill melinda gate foundat a wealth of eclect restaur and bar make thi area of seattl one of the most sought out by local and visitor our proxim to lake union allow visitor to take in some of the pacif northwest majest sceneri and enjoy outdoor activ like kayak and sail over 2000 sq ft of versatil space and a complimentari busi center stateoftheart av technolog and our help staff will guarante your confer cocktail recept or wed is a success refresh in the sparkl saltwat pool or energ with the latest equip in the 24hour fit center tast decor and flood with natur light our guest room and suit offer everyth you need to relax and stay product unwind in the bar and enjoy american cuisin for breakfast lunch and dinner in our restaur the 24hour pavilion pantri stock a varieti of snack drink and sundri
1,Sheraton Grand Seattle,"Located in the city's vibrant core, the Sheraton Grand Seattle provides a gateway to the diverse sights and sounds of the Pacific Northwest. Step out of our front doors to find gourmet dining and bars, world-class shopping, exciting entertainment, and iconic local attractions including the Pike Place Market, Space Needle and Chihuly Garden & Glass Museum. As one of only seven Sheraton hotels in North America to earn the esteemed Grand designation, guests can book confidently knowing they?re receiving the highest benchmark on product and service offerings available. Experience our recently completed multimillion-dollar transformation featuring all new guest rooms, an expanded Sheraton Club Lounge, and modern meeting & event spaces. Gather in our stylish new lobby and enjoy our private art collection featuring local artists while enjoying your favorite beverage from Starbucks. The Sheraton Grand features several dining options including Loulay Kitchen & Bar by James Beard award winning chef Thierry Rautureau.",locat in the citi vibrant core the sheraton grand seattl provid a gateway to the divers sight and sound of the pacif northwest step out of our front door to find gourmet dine and bar worldclass shop excit entertain and icon local attract includ the pike place market space needl and chihuli garden glass museum as one of onli seven sheraton hotel in north america to earn the esteem grand design guest can book confid know theyr receiv the highest benchmark on product and servic offer avail experi our recent complet multimilliondollar transform featur all new guest room an expand sheraton club loung and modern meet event space gather in our stylish new lobbi and enjoy our privat art collect featur loca

#### Display updated dataframe

In [ ]:
df.set_index('name', inplace=True)
df.head()

,desc,desc_clean
name,,
Hilton Garden Seattle Downtown,"Located on the southern tip of Lake Union, the Hilton Garden Inn Seattle Downtown hotel is perfectly located for business and leisure. \nThe neighborhood is home to numerous major international companies including Amazon, Google and the Bill & Melinda Gates Foundation. A wealth of eclectic restaurants and bars make this area of Seattle one of the most sought out by locals and visitors. Our proximity to Lake Union allows visitors to take in some of the Pacific Northwest's majestic scenery and enjoy outdoor activities like kayaking and sailing. over 2,000 sq. ft. of versatile space and a complimentary business center. State-of-the-art A/V technology and our helpful staff will guarantee your conference, cocktail reception or wedding is a success. Refresh in the sparkling saltwater pool, or energize with the latest equipment in the 24-hour fitness center. Tastefully decorated and flooded with natural light, our guest rooms and suites offer everything you need to relax and stay productive. Unwind in the bar, and enjoy American cuisine for breakfast, lunch and dinner in our restaurant. The 24-hour Pavilion Pantry? stocks a variety of snacks, drinks and sundries.",locat on the southern tip of lake union the hilton garden inn seattl downtown hotel is perfectli locat for busi and leisur the neighborhood is home to numer major intern compani includ amazon googl and the bill melinda gate foundat a wealth of eclect restaur and bar make thi area of seattl one of the most sought out by local and visitor our proxim to lake union allow visitor to take in some of the pacif northwest majest sceneri and enjoy outdoor activ like kayak and sail over 2000 sq ft of versatil space and a complimentari busi center stateoftheart av technolog and our help staff will guarante your confer cocktail recept or wed is a success refresh in the sparkl saltwat pool or energ with the latest equip in the 24hour fit center tast decor and flood with natur light our guest room and suit offer everyth you need to relax and stay product unwind in the bar and enjoy american cuisin for breakfast lunch and dinner in our restaur the 24hour pavilion pantri stock a varieti of snack drink and sundri
Sheraton Grand Seattle,"Located in the city's vibrant core, the Sheraton Grand Seattle provides a gateway to the diverse sights and sounds of the Pacific Northwest. Step out of our front doors to find gourmet dining and bars, world-class shopping, exciting entertainment, and iconic local attractions including the Pike Place Market, Space Needle and Chihuly Garden & Glass Museum. As one of only seven Sheraton hotels in North America to earn the esteemed Grand designation, guests can book confidently knowing they?re receiving the highest benchmark on product and service offerings available. Experience our recently completed multimillion-dollar transformation featuring all new guest rooms, an expanded Sheraton Club Lounge, and modern meeting & event spaces. Gather in our stylish new lobby and enjoy our private art collection featuring local artists while enjoying your favorite beverage from Starbucks. The Sheraton Grand features several dining options including Loulay Kitchen & Bar by James Beard award winning chef Thierry Rautureau.",locat in the citi vibrant core the sheraton grand seattl provid a gateway to the divers sight and sound of the pacif northwest step out of our front door to find gourmet dine and bar worldclass shop excit entertain and icon local attract includ the pike place market space needl and chihuli garden glass museum as one of onli seven sheraton hotel in north america to earn the esteem grand design guest can book confid know theyr receiv the highest benchmark on product and servic offer avail experi our recent complet multimilliondollar transform featur all new guest room an expand sheraton club loung and modern meet event space gather in our stylish new lobbi and enjoy our privat art collect featur local 

### Vectorize cleaned hotel descriptions

In [ ]:
# Instantiate TF-IDF vectorizer
#  Notice the ngram_range
tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 3), stop_words='english')

In [ ]:
# Create a hotel descriptions matrix for each ngram and its TF-IDF score with regard to each hotel description
tfidf_matrix = tf.fit_transform(df['desc_clean'])
tfidf_matrix

<152x26084 sparse matrix of type '<class 'numpy.float64'>'
	with 39845 stored elements in Compressed Sparse Row format>

### Generate similarities matrix on cleaned hotel descriptions

In [ ]:
# cosine_similarities = cosine_similarity(tfidf_matrix, tfidf_matrix)
cosine_similarities = cosine_similarity(tfidf_matrix)
cosine_similarities.shape

(152, 152)

In [ ]:
cosine_similarities

array([[1.        , 0.01794847, 0.03479381, ..., 0.01363881, 0.00414717,
        0.00990616],
       [0.01794847, 1.        , 0.01870858, ..., 0.01878472, 0.00681159,
        0.00947376],
       [0.03479381, 0.01870858, 1.        , ..., 0.02936464, 0.0127602 ,
        0.0106586 ],
       ...,
       [0.01363881, 0.01878472, 0.02936464, ..., 1.        , 0.0134766 ,
        0.00820755],
       [0.00414717, 0.00681159, 0.0127602 , ..., 0.0134766 , 1.        ,
        0.00587665],
       [0.00990616, 0.00947376, 0.0106586 , ..., 0.00820755, 0.00587665,
        1.        ]])

In [ ]:
# Save hotel indices for recommendation lookups
hotel_indices = pd.Series(df.index)
hotel_indices

,name
0,Hilton Garden Seattle Downtown
1,Sheraton Grand Seattle
2,Crowne Plaza Seattle Downtown
3,Kimpton Hotel Monaco Seattle
4,The Westin Seattle
...,...
147,The Halcyon Suite Du Jour
148,Vermont Inn
149,Stay Alfred on Wall Street
150,Pike's Place Lux Suites by Barsala


### Create hotel recommender

In [ ]:
def recommendations(seed_hotel_name, hotels=df, cosine_similarities=cosine_similarities):

    recommended_hotels = []

    # Get hotel index of the seed hotel name
    seed_hotel_index = hotel_indices[hotel_indices == seed_hotel_name].index[0]

    # Create a Series with the similarity scores in descending order
    score_series = pd.Series(cosine_similarities[seed_hotel_index]).sort_values(ascending = False)

    # Get the indexes of the 5 most similar hotels except itself
    top_indexes = list(score_series.iloc[1:6].index)

    # Populate the list with the names of the top 5 matching hotels
    count = 0
    recommended_hotels.append(f"Seed hotel : {seed_hotel_name}")
    for i in top_indexes:
        count += 1
        hotel = list(hotels.index)[i]
        recommended_hotels.append(f"Recommendation {count} : {hotel}")

    return recommended_hotels

### Make hotel recommendations for the following hotel names:
* Motel 6 Seattle Sea-Tac Airport South
* The Bacon Mansion Bed and Breakfast
* Holiday Inn Seattle Downtown

In [ ]:
recommendations('Motel 6 Seattle Sea-Tac Airport South')

['Seed hotel : Motel 6 Seattle Sea-Tac Airport South',
 'Recommendation 1 : Emerald Motel',
 'Recommendation 2 : Ramada by Wyndham SeaTac Airport',
 'Recommendation 3 : DoubleTree by Hilton Hotel Seattle Airport',
 'Recommendation 4 : Red Roof Inn Seattle Airport - SEATAC',
 'Recommendation 5 : Country Inn & Suites by Radisson, Seattle-Tacoma International Airport']

In [ ]:
recommendations("The Bacon Mansion Bed and Breakfast")

['Seed hotel : The Bacon Mansion Bed and Breakfast',
 'Recommendation 1 : 11th Avenue Inn Bed and Breakfast',
 'Recommendation 2 : Shafer Baillie Mansion Bed & Breakfast',
 'Recommendation 3 : Gaslight Inn',
 'Recommendation 4 : Chittenden House Bed and Breakfast',
 'Recommendation 5 : Silver Cloud Hotel - Seattle Broadway']

In [ ]:
recommendations("Holiday Inn Seattle Downtown")

['Seed hotel : Holiday Inn Seattle Downtown',
 'Recommendation 1 : Holiday Inn Express & Suites Seattle-City Center',
 'Recommendation 2 : Inn at Queen Anne',
 'Recommendation 3 : Holiday Inn Express & Suites North Seattle - Shoreline',
 'Recommendation 4 : Hotel Theodore',
 'Recommendation 5 : Silver Cloud Hotel - Seattle Stadium']